# Teacher labelling — draft invoice tables with the Qwen3.6-35B-A3B-FP8 teacher

**Company server, PRIVATE data.** The 20 confidential invoices never leave the
network. The on-prem teacher (Qwen3.6-35B-A3B-FP8, vision + thinking) drafts a
first logical-HTML label for each; a human corrects them into the eval set the
project lacks. Two-stage: stage-1 OCR+geometry grounds a stage-2 schema-conditioned
reasoning pass.

**One instruction everywhere.** `INSTRUCTION` below must be byte-identical in this
notebook, `two-stage-reconstruct.ipynb`, and `finetune-and-serve.ipynb` — a call
that builds its own variant (different `with_bbox`, different schema hint) is the
bug that made `data-bbox` appear on one call and vanish on the next, and it breaks
the train/inference prompt-match rule.

Nothing here calls a hosted API — we serve the teacher locally with vLLM and hit
loopback.

## 0. Serve the teacher (run once, in a terminal on the box)

```bash
vllm serve Qwen/Qwen3.6-35B-A3B-FP8 --port 8000 \
    --no-enable-prefix-caching --limit-mm-per-prompt image=4
```
The FP8 checkpoint fits the L40's 48 GB (3B active params make it fast).
`--no-enable-prefix-caching`: a labelling pass re-sends near-identical prompts, and
multimodal prefix-cache hits have degraded repeat-call outputs on some vLLM
versions. `image=4` lets one long table arrive as a list of page crops. Leave it
running; the cells below hit `http://localhost:8000/v1`.

## 1. Config

In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path

from src.model.registry import MODEL_QWEN36_35B_FP8
from src.model.vllm_client import VLLMTableReconstructor
from src.model.prompts import build_schema_instruction, DEFAULT_INVOICE_SCHEMA
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout

IMAGES_DIR = Path("data/invoices")          # the 20 confidential images (never synced out)
OUT = Path("data/invoices/teacher_labels.jsonl")
IMAGE_GLOB = ("*.png", "*.jpg", "*.jpeg")

# THE one instruction — identical in two-stage-reconstruct.ipynb and
# finetune-and-serve.ipynb (train/inference prompts must match, and with_bbox
# must be the same on every call or boxes appear on one call and not the next).
# Document-agnostic by default; add a header hint ONLY if the family's header is
# known stable:  build_schema_instruction(DEFAULT_INVOICE_SCHEMA, with_bbox=True)
INSTRUCTION = build_schema_instruction(with_bbox=True)

teacher = VLLMTableReconstructor(model_id=MODEL_QWEN36_35B_FP8, base_url="http://localhost:8000/v1", thinking=True)
print("teacher:", teacher.model_id)

## 2. Stage 1 — OCR + geometry (grounding), eyeballed on one image

In [ ]:
images = sorted(p for pat in IMAGE_GLOB for p in IMAGES_DIR.glob(pat))
print(f"{len(images)} invoices")

sample = images[0]
layout = serialize_layout(run_ocr(sample), style="grid")
print(layout[:1200])

## 3. Stage 2 — schema-conditioned reconstruction on that image

In [ ]:
pred = teacher.predict(sample, instruction=INSTRUCTION, ocr_layout=layout)
print(pred.html[:2000] if pred.html else "*** empty — check the served endpoint / prompt ***")

## 4. Render it next to the image — the only quality gate you have

No ground truth exists yet, so a human reads every draft. Totals reconciliation
(`sum(line_items) ≈ subtotal`, `+tax ≈ total`) is the cheapest automatic flag.

In [ ]:
from IPython.display import HTML, Image as IPyImage, display
display(IPyImage(filename=str(sample), width=520))
display(HTML(pred.html or "<i>empty</i>"))

## 5. Batch — label every invoice (resumable)

In [ ]:
import json

def done_uids(path):
    if not path.exists():
        return set()
    return {json.loads(l)["uid"] for l in path.read_text().splitlines() if l.strip()}

already = done_uids(OUT)
OUT.parent.mkdir(parents=True, exist_ok=True)
with OUT.open("a") as f:
    for i, img in enumerate(images, 1):
        uid = img.stem
        if uid in already:
            continue
        layout = serialize_layout(run_ocr(img), style="grid")
        pred = teacher.predict(img, instruction=INSTRUCTION, ocr_layout=layout)
        f.write(json.dumps({"uid": uid, "image": str(img), "html": pred.html}) + "\n")
        print(f"  {i}/{len(images)} {uid}: {len(pred.html)} chars")
print("labels ->", OUT)

---
**Next:** human-correct `teacher_labels.jsonl` in place, then `finetune-and-serve.ipynb`
distils an 8B student from the corrected labels and serves it with vLLM.